# RUPA-DSA v0 — train UCF-Crime / XD-Violence on Kaggle

This notebook implements the narrow hypothesis **video-specific counterfactual normal reconstruction with category-aware residual semantic alignment**:

```text
video-specific DNP -> F_rec -> R = F_video - F_rec
                      |             |
                 normal text   anomaly-category text
```

Run one benchmark per Kaggle GPU session. The notebook accepts extracted `.npy` features or the original ZIP archives, supports resume, runs a small GPU forward/backward preflight, and packages the best weights plus metrics.

Kaggle settings: enable **GPU** and **Internet**. Attach `UCFClipFeatures.zip` for UCF, or both `XDTrainClipFeatures.zip` and `XDTestClipFeatures.zip` for XD.

In [ ]:
# ========================= USER CONFIG =========================
DATASET = "ucf"              # "ucf" or "xd"; use a separate session for each
MAX_EPOCHS = 10               # total target epoch count, including resumed epochs
RESUME = False                # attach an earlier result/checkpoint and set True
RUN_PREFLIGHT = True          # recommended: catches shape/NaN/GPU issues before training

UCF_BATCH_SIZE = 16            # T4/P100 16 GB; reduce to 8 on CUDA OOM
XD_BATCH_SIZE = 24             # reduce to 16 or 8 on CUDA OOM
NUM_WORKERS = 2
SEED = 234

# Closed-loop routing: normalized automatically by the model.
ROUTING_DET_WEIGHT = 0.5
ROUTING_REC_WEIGHT = 0.3
ROUTING_SEM_WEIGHT = 0.2

# RUPA loss ablations. Keep these defaults for the full model.
LOSS_RESIDUAL_WEIGHT = 1.0
LOSS_RECONSTRUCTED_NORMAL_WEIGHT = 1.0
LOSS_DNP_NORMAL_WEIGHT = 0.1
LOSS_CONSISTENCY_WEIGHT = 1.0
LOSS_GATHER_WEIGHT = 1.0

REPO_URL = "https://github.com/Sharvuz/RUPA-DSA-v0.git"
REPO_REF = None                # optional commit/tag for a reproducible baseline

assert DATASET in {"ucf", "xd"}
assert MAX_EPOCHS > 0
assert sum([ROUTING_DET_WEIGHT, ROUTING_REC_WEIGHT, ROUTING_SEM_WEIGHT]) > 0
print("RUPA-DSA v0 config:", DATASET, MAX_EPOCHS, "epochs", "resume=", RESUME)

## 1. Clone RUPA-DSA v0 and prepare the GPU environment

Kaggle clones the source directly from `Sharvuz/RUPA-DSA-v0`. No source code or base64 overlay is embedded in this notebook. Set `REPO_REF` in the config cell when you want to pin an exact branch or tag.

In [ ]:
import json, os, random, shutil, subprocess, sys, time, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "RUPA-DSA-v0"
ARTIFACTS = WORK / "rupa_v0_artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

if REPO.exists():
    shutil.rmtree(REPO)
clone_cmd = ["git", "clone", "--depth", "1"]
if REPO_REF:
    clone_cmd += ["--branch", REPO_REF]
clone_cmd += [REPO_URL, str(REPO)]
subprocess.run(clone_cmd, check=True)
os.chdir(REPO)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "ftfy", "regex", "einops==0.8.0", "ipdb", "scikit-learn", "pandas"
], check=True)

import numpy as np
import pandas as pd
import torch

print("Repository:", REPO_URL, "ref:", REPO_REF or "default branch")
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Enable Accelerator = GPU in Kaggle Notebook Settings."
assert "RUPA-DSA" in (REPO / "src/model.py").read_text(encoding="utf-8")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("Direct GitHub clone ready.")

## 2. Find/extract features and rebuild CSV paths

No Kaggle Dataset slug or mount name is hard-coded. ZIP extraction is only used when the required `.npy` files are not already mounted.

In [ ]:
from collections import defaultdict
from pathlib import PurePosixPath

if DATASET == "ucf":
    csv_specs = [
        ("list/ucf_CLIP_rgb.csv", "ucf_train.csv"),
        ("list/ucf_CLIP_rgbtest.csv", "ucf_test.csv"),
    ]
    zip_hints = ["ucfclipfeatures"]
else:
    csv_specs = [
        ("list/xd_CLIP_rgb.csv", "xd_train.csv"),
        ("list/xd_CLIP_rgbtest.csv", "xd_test.csv"),
    ]
    zip_hints = ["xdtrainclipfeatures", "xdtestclipfeatures"]

def basename(path_string):
    return PurePosixPath(str(path_string).replace("\\", "/")).name

required_names = set()
for csv_rel, _ in csv_specs:
    frame = pd.read_csv(REPO / csv_rel)
    required_names.update(basename(p) for p in frame["path"])

feature_roots = [INPUT]

def indexed_names(roots):
    return {p.name for root in roots for p in root.rglob("*.npy")}

missing_before = required_names - indexed_names(feature_roots)
if missing_before:
    extract_root = WORK / f"rupa_features_{DATASET}"
    extract_root.mkdir(parents=True, exist_ok=True)
    archives = [
        p for p in INPUT.rglob("*.zip")
        if any(hint in p.stem.lower() for hint in zip_hints)
    ]
    if archives:
        print(f"Extracting {len(archives)} feature archive(s); this can take several minutes...")
    for archive_path in archives:
        destination = (extract_root / archive_path.stem).resolve()
        destination.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive_path) as archive:
            for member in archive.infolist():
                target = (destination / member.filename).resolve()
                if destination not in target.parents and target != destination:
                    raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
            archive.extractall(destination)
        print("Extracted:", archive_path.name)
    feature_roots.append(extract_root)

index = defaultdict(list)
for root in feature_roots:
    for path in root.rglob("*.npy"):
        if path.name in required_names:
            index[path.name].append(path)

def rank_candidate(candidate, original):
    original_parts = [x.lower() for x in str(original).replace("\\", "/").split("/")[-4:]]
    candidate_text = str(candidate).lower()
    return sum(part in candidate_text for part in original_parts)

def rewrite_csv(csv_rel, output_name):
    frame = pd.read_csv(REPO / csv_rel)
    resolved, missing, ambiguous = [], [], 0
    for original in frame["path"].astype(str):
        matches = index.get(basename(original), [])
        if not matches:
            missing.append(basename(original))
            resolved.append("")
            continue
        matches = sorted(matches, key=lambda p: rank_candidate(p, original), reverse=True)
        resolved.append(str(matches[0]))
        ambiguous += int(len(matches) > 1)
    if missing:
        raise FileNotFoundError(
            f"{csv_rel}: missing {len(missing)}/{len(frame)} features; examples={missing[:10]}"
        )
    output = ARTIFACTS / output_name
    frame["path"] = resolved
    frame.to_csv(output, index=False)
    print(f"{output_name}: rows={len(frame)}, missing=0, duplicate basenames={ambiguous}")
    return output

train_csv = rewrite_csv(*csv_specs[0])
test_csv = rewrite_csv(*csv_specs[1])

sample_path = Path(pd.read_csv(train_csv).iloc[0]["path"])
sample = np.load(sample_path, mmap_mode="r")
assert sample.ndim == 2 and sample.shape[-1] == 512, (
    f"Expected CLIP feature [T,512], got {sample.shape} from {sample_path}"
)
print("Feature preflight:", sample_path.name, sample.shape)

## 3. GPU forward/backward preflight

This is a small real RUPA pass (not a mock): it verifies `F_rec`, `R`, semantic logits, fused scores, finite losses, and gradients before a long Kaggle run.

In [ ]:
if RUN_PREFLIGHT:
    sys.path.insert(0, str(REPO / "src"))
    from model import DSANet
    from utils.tools import get_prompt_text
    if DATASET == "ucf":
        import ucf_option as dataset_option
        label_map = {
            'Normal': 'normal', 'Abuse': 'abuse', 'Arrest': 'arrest',
            'Arson': 'arson', 'Assault': 'assault', 'Burglary': 'burglary',
            'Explosion': 'explosion', 'Fighting': 'fighting',
            'RoadAccidents': 'roadAccidents', 'Robbery': 'robbery',
            'Shooting': 'shooting', 'Shoplifting': 'shoplifting',
            'Stealing': 'stealing', 'Vandalism': 'vandalism'
        }
    else:
        import xd_option as dataset_option
        label_map = {
            'A': 'normal', 'B1': 'fighting', 'B2': 'shooting',
            'B4': 'riot', 'B5': 'abuse', 'B6': 'car accident', 'G': 'explosion'
        }

    smoke_args = dataset_option.parser.parse_args([])
    smoke_args.visual_length = 16
    smoke_args.visual_layers = 1
    smoke_args.attn_window = 8
    smoke_args.decoder_depth = 1
    smoke_args.num_prototypes = 4
    smoke_args.normal_selection_ratio = 0.25
    smoke_args.routing_det_weight = ROUTING_DET_WEIGHT
    smoke_args.routing_rec_weight = ROUTING_REC_WEIGHT
    smoke_args.routing_sem_weight = ROUTING_SEM_WEIGHT
    smoke_args.rupa_use = True

    device = "cuda"
    smoke_model = DSANet(
        smoke_args.classes_num, smoke_args.embed_dim, smoke_args.visual_length,
        smoke_args.visual_width, smoke_args.visual_head, smoke_args.visual_layers,
        smoke_args.attn_window, smoke_args.prompt_prefix, smoke_args.prompt_postfix,
        smoke_args, device
    ).to(device).train()
    smoke_visual = torch.randn(2, 16, 512, device=device)
    smoke_lengths = torch.tensor([16, 12], device=device)
    outputs = smoke_model(
        smoke_visual, None, get_prompt_text(label_map), smoke_lengths, True
    )
    text_features, logits1, logits2, logits3, logits4, dnp = outputs
    expected_keys = {
        "reconstructed_features", "residual_features", "dynamic_normal_patterns",
        "semantic_logits", "routing_scores", "routing_logits", "dnp_normal_loss"
    }
    assert expected_keys.issubset(dnp)
    assert torch.allclose(
        dnp["residual_features"],
        dnp["original_features"] - dnp["reconstructed_features"],
        atol=1e-5,
    )
    assert dnp["semantic_logits"].shape == (2, 16, smoke_args.classes_num)
    assert torch.all((dnp["routing_scores"] > 0) & (dnp["routing_scores"] < 1))
    smoke_loss = (
        logits1.square().mean() + logits2.square().mean() * 1e-4
        + logits3.square().mean() * 1e-4 + logits4.square().mean() * 1e-4
        + dnp["g_loss"] + dnp["dnp_normal_loss"]
    )
    assert torch.isfinite(smoke_loss)
    smoke_loss.backward()
    assert any(
        p.grad is not None and torch.isfinite(p.grad).all()
        for p in smoke_model.video_anomaly_refiner.parameters()
    )
    print("RUPA GPU preflight: OK | loss=", float(smoke_loss.detach()))
    del smoke_model, smoke_visual, outputs, dnp
    torch.cuda.empty_cache()
else:
    print("RUPA GPU preflight skipped by config.")

## 4. Resume (optional) and train

`MAX_EPOCHS` is the total target. For example, a checkpoint saved at epoch 4 resumes at epoch 5 and stops at epoch 10. Logs stream live and are also saved.

In [ ]:
dataset_dir = ARTIFACTS / DATASET
dataset_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = dataset_dir / f"checkpoint_{DATASET}.pth"
model_path = dataset_dir / f"best_{DATASET}.pth"

def find_resume_checkpoint():
    direct = list(INPUT.rglob(f"checkpoint_{DATASET}.pth"))
    if direct:
        return direct[0]
    for archive_path in INPUT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(archive_path) as archive:
                matches = [
                    name for name in archive.namelist()
                    if name.endswith(f"checkpoint_{DATASET}.pth")
                ]
                if matches:
                    with archive.open(matches[0]) as src, checkpoint_path.open("wb") as dst:
                        shutil.copyfileobj(src, dst)
                    return checkpoint_path
        except zipfile.BadZipFile:
            continue
    return None

resume_args = []
if RESUME:
    resume_source = find_resume_checkpoint()
    if resume_source is None:
        raise FileNotFoundError(
            f"RESUME=True but checkpoint_{DATASET}.pth was not found under /kaggle/input"
        )
    if Path(resume_source).resolve() != checkpoint_path.resolve():
        shutil.copy2(resume_source, checkpoint_path)
    resume_args = ["--use-checkpoint", "true"]
    print("Resume checkpoint:", resume_source)

batch_size = UCF_BATCH_SIZE if DATASET == "ucf" else XD_BATCH_SIZE
cmd = [
    sys.executable, f"src/{DATASET}_train.py",
    "--train-list", str(train_csv),
    "--test-list", str(test_csv),
    "--model-path", str(model_path),
    "--checkpoint-path", str(checkpoint_path),
    "--max-epoch", str(MAX_EPOCHS),
    "--batch-size", str(batch_size),
    "--num-workers", str(NUM_WORKERS),
    "--seed", str(SEED),
    "--rupa-use", "true",
    "--routing-det-weight", str(ROUTING_DET_WEIGHT),
    "--routing-rec-weight", str(ROUTING_REC_WEIGHT),
    "--routing-sem-weight", str(ROUTING_SEM_WEIGHT),
    "--loss-residual-weight", str(LOSS_RESIDUAL_WEIGHT),
    "--loss-reconstructed-normal-weight", str(LOSS_RECONSTRUCTED_NORMAL_WEIGHT),
    "--loss-dnp-normal-weight", str(LOSS_DNP_NORMAL_WEIGHT),
    "--loss-consistency-weight", str(LOSS_CONSISTENCY_WEIGHT),
    "--loss-gather-weight", str(LOSS_GATHER_WEIGHT),
] + resume_args

config = {
    "dataset": DATASET, "max_epochs": MAX_EPOCHS, "resume": RESUME,
    "batch_size": batch_size, "num_workers": NUM_WORKERS, "seed": SEED,
    "routing_weights": [ROUTING_DET_WEIGHT, ROUTING_REC_WEIGHT, ROUTING_SEM_WEIGHT],
    "loss_weights": {
        "residual": LOSS_RESIDUAL_WEIGHT,
        "reconstructed_normal": LOSS_RECONSTRUCTED_NORMAL_WEIGHT,
        "dnp_normal": LOSS_DNP_NORMAL_WEIGHT,
        "consistency": LOSS_CONSISTENCY_WEIGHT,
        "gather": LOSS_GATHER_WEIGHT,
    },
    "repo_url": REPO_URL, "repo_ref": REPO_REF,
    "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0),
    "command": cmd,
}
(dataset_dir / "run_config.json").write_text(
    json.dumps(config, indent=2), encoding="utf-8"
)
print("Train command:", " ".join(cmd))

log_path = dataset_dir / f"train_{DATASET}.log"
started = time.time()
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
with log_path.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        cmd, cwd=REPO, env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
        log_file.flush()
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Training failed with exit code {return_code}; see {log_path}")

elapsed_hours = (time.time() - started) / 3600
(dataset_dir / "runtime_hours.txt").write_text(
    f"{elapsed_hours:.4f}\n", encoding="utf-8"
)
assert checkpoint_path.exists() and model_path.exists()
print(f"Training complete in {elapsed_hours:.2f} hours")

## 5. Evaluate the best checkpoint and package artifacts

The final ZIP contains the best weights, resumable checkpoint, logs, mapped CSV files, run config, metrics, and the exact RUPA source used.

In [ ]:
test_cmd = [
    sys.executable, f"src/{DATASET}_test.py",
    "--test-list", str(test_csv),
    "--model-path", str(model_path),
    "--rupa-use", "true",
    "--routing-det-weight", str(ROUTING_DET_WEIGHT),
    "--routing-rec-weight", str(ROUTING_REC_WEIGHT),
    "--routing-sem-weight", str(ROUTING_SEM_WEIGHT),
]
result = subprocess.run(
    test_cmd, cwd=REPO, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print(result.stdout)
(dataset_dir / f"metrics_{DATASET}.txt").write_text(
    result.stdout, encoding="utf-8"
)
if result.returncode != 0:
    raise RuntimeError("Best-checkpoint evaluation failed; inspect the output above.")

shutil.copy2(train_csv, dataset_dir / train_csv.name)
shutil.copy2(test_csv, dataset_dir / test_csv.name)
source_snapshot = dataset_dir / "source"
source_snapshot.mkdir(exist_ok=True)
for relative in [
    "src/model.py", f"src/{DATASET}_train.py", f"src/{DATASET}_test.py",
    f"src/{DATASET}_option.py", "RUPA_DSA.md",
]:
    source_file = REPO / relative
    destination = source_snapshot / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_file, destination)

archive = shutil.make_archive(
    str(ARTIFACTS / f"RUPA_DSA_v0_{DATASET}_results"),
    "zip", root_dir=dataset_dir,
)
print("DONE:", archive)
print("Refresh the Kaggle Files panel and download this ZIP.")

## Experiment discipline

For a defensible result, keep the seed/epochs/batch size fixed and run at least these controls:

1. DSANet control: change the train/test command to `--rupa-use false`.
2. No semantic routing: `(w_det, w_rec, w_sem) = (0.7, 0.3, 0.0)`.
3. Full RUPA: `(0.5, 0.3, 0.2)`.
4. No DNP-normal alignment: `LOSS_DNP_NORMAL_WEIGHT = 0.0`.
5. No residual-event alignment: `LOSS_RESIDUAL_WEIGHT = 0.0`.

Run at least three seeds and report mean ± standard deviation. Do not compare a single RUPA run against a cherry-picked baseline checkpoint.